# Part 20: Agentic AI Foundations & OpenAI Agents SDK

> From basic LLM API usage to full agentic systems: design patterns, tool loops, multi-model support, and the OpenAI Agents SDK.

---


## 20.1 What Makes an AI "Agentic"?

| Characteristic | Description |
|---------------|-------------|
| **Goal-directed** | Works toward an objective, not just one response |
| **Tool use** | Can call external APIs, run code, search web |
| **Memory** | Maintains context across steps |
| **Planning** | Decides what to do next dynamically |
| **Action** | Takes real-world actions (send email, book meeting) |

### Fundamental Shift
- **LLM (one-shot):** Input → Output
- **Agent (loop):** Input → Think → Act → Observe → Think → Act → ... → Output

### The Agent Loop

![Agent Loop](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_agent_loop.png)

The agent continuously:
1. Receives input from **Human**
2. Makes an **LLM Call** (think + decide)
3. Takes an **Action** on the **Environment** (tool call, API, file system)
4. Gets **Feedback** from the environment
5. Repeats until a **STOP** condition is met

### How Tools Connect Code ↔ LLM

![Tool Execution](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_tool_execution.png)

Code sends a **Prompt** to the LLM and receives a **Response**. The LLM may also request **tool execution** — the code runs the tool and returns results back to the LLM.

## 20.2 The 6 Core Agentic Design Patterns

### Available Frameworks

![Frameworks](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_frameworks.png)

You can implement these patterns with **any** of these frameworks — or with no framework at all (direct API calls).

---

### Pattern 1: Prompt Chaining (Pipeline)

![Pipeline](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_pipeline.png)

**IN → LLM1 → Gate → LLM2 → LLM3 → OUT**

Each LLM call's output feeds the next. A **Gate** can add conditional logic — only proceed if quality passes.

In [ ]:
from openai import OpenAI
client = OpenAI()

def chain_prompts(initial_input: str) -> str:
    """Sequential LLM calls — each output feeds the next."""

    # Step 1: Extract key business requirements
    step1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract the key business requirements as bullet points."},
            {"role": "user",   "content": initial_input}
        ]
    ).choices[0].message.content

    # Step 2: Design a solution from requirements
    step2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Design a technical solution for these requirements."},
            {"role": "user",   "content": step1}
        ]
    ).choices[0].message.content

    # Step 3: Write implementation plan from solution
    step3 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Write a detailed implementation plan."},
            {"role": "user",   "content": step2}
        ]
    ).choices[0].message.content

    return step3

### Pattern 2: Router

![Router](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_router.png)

**IN → LLM Router → LLM1 / LLM2 / LLM3 → OUT**

An LLM (or rule) examines the input and routes it to the most appropriate specialist model.

In [ ]:
from openai import OpenAI
client = OpenAI()

# Specialist system prompts
SPECIALISTS = {
    "technical": "You are a technical expert. Answer technical questions about code and software.",
    "billing":   "You are a billing specialist. Handle payments, invoices, and subscription questions.",
    "general":   "You are a helpful general assistant.",
}

def route_and_answer(user_query: str) -> str:
    """Route query to the right specialist via an LLM router."""

    # Router LLM decides which specialist to use
    routing = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Classify this query into exactly one category: technical, billing, or general.
Reply with only the category word.

Query: {user_query}"""
        }]
    ).choices[0].message.content.strip().lower()

    specialist = SPECIALISTS.get(routing, SPECIALISTS["general"])
    print(f"Routed to: {routing}")

    # Specialist LLM answers
    answer = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": specialist},
            {"role": "user",   "content": user_query}
        ]
    ).choices[0].message.content

    return answer

# print(route_and_answer("My invoice shows wrong amount"))
# print(route_and_answer("How do I implement a binary search tree?"))

### Pattern 3: Parallelization (Coordinator + Aggregator)

![Parallelization](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_parallelization.png)

**IN → Coordinator → [LLM1 ‖ LLM2 ‖ LLM3] → Aggregator → OUT**

Multiple LLMs work simultaneously on different sub-tasks. Results are aggregated into a final answer.

In [ ]:
import asyncio
from openai import OpenAI
client = OpenAI()

async def parallel_analysis(topic: str) -> dict:
    """Run multiple analyses simultaneously, then aggregate."""

    async def analyze(perspective: str) -> str:
        response = await asyncio.to_thread(
            client.chat.completions.create,
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"Analyze from a {perspective} perspective in 2-3 sentences."},
                {"role": "user",   "content": topic}
            ]
        )
        return response.choices[0].message.content

    # Coordinator: launch all analyses in parallel
    technical, business, risk = await asyncio.gather(
        analyze("technical"),
        analyze("business"),
        analyze("risk"),
    )

    # Aggregator: synthesize results
    combined = f"Technical: {technical}\n\nBusiness: {business}\n\nRisk: {risk}"
    summary = await asyncio.to_thread(
        client.chat.completions.create,
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Synthesize these perspectives into a concise executive summary."},
            {"role": "user",   "content": combined}
        ]
    )

    return {
        "technical": technical,
        "business":  business,
        "risk":      risk,
        "summary":   summary.choices[0].message.content
    }

# result = asyncio.run(parallel_analysis("Migrating a monolith to microservices"))
# print(result["summary"])

### Pattern 4: Orchestrator-Workers (Synthesizer)

![Orchestrator](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_orchestrator.png)

**IN → Orchestrator → [Worker LLM1 ‖ Worker LLM2 ‖ Worker LLM3] → Synthesizer → OUT**

The **Orchestrator** breaks the task into sub-tasks and assigns them to worker LLMs. The **Synthesizer** (same or different LLM) combines sub-results into a final output.

In [ ]:
import asyncio, json
from openai import OpenAI
client = OpenAI()

async def orchestrate_task(complex_task: str) -> str:
    """Orchestrator breaks task into sub-tasks, workers execute, synthesizer combines."""

    # Orchestrator: decompose task
    plan_response = await asyncio.to_thread(
        client.chat.completions.create,
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Break this task into exactly 3 parallel sub-tasks.
Return JSON: {{"subtasks": ["subtask1", "subtask2", "subtask3"]}}

Task: {complex_task}"""
        }]
    )
    plan = json.loads(plan_response.choices[0].message.content)
    subtasks = plan["subtasks"]
    print(f"Orchestrator created {len(subtasks)} sub-tasks")

    # Workers: execute sub-tasks in parallel
    async def worker(subtask: str) -> str:
        resp = await asyncio.to_thread(
            client.chat.completions.create,
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a specialist. Complete your assigned sub-task thoroughly."},
                {"role": "user",   "content": subtask}
            ]
        )
        return resp.choices[0].message.content

    results = await asyncio.gather(*[worker(st) for st in subtasks])

    # Synthesizer: combine worker results
    combined = "\n\n".join(f"Sub-task {i+1} result:\n{r}" for i, r in enumerate(results))
    synthesis = await asyncio.to_thread(
        client.chat.completions.create,
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Synthesize these sub-task results into a cohesive final answer."},
            {"role": "user",   "content": f"Original task: {complex_task}\n\n{combined}"}
        ]
    )
    return synthesis.choices[0].message.content

# result = asyncio.run(orchestrate_task("Write a comprehensive market analysis for electric vehicles"))
# print(result)

### Pattern 5: Evaluation & Re-running (Generator-Evaluator)

![Evaluator](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_evaluator.png)

**IN → LLM Generator → LLM Evaluator → (Accepted: OUT) | (Rejected: feedback → LLM Generator again)**

One LLM generates, another evaluates. If rejected, feedback is sent back to regenerate. This loop continues until quality threshold is met.

In [ ]:
from pydantic import BaseModel
from openai import OpenAI
client = OpenAI()

class EvalResult(BaseModel):
    score: int       # 1–10
    passed: bool
    feedback: str

def generate_with_quality_gate(task: str, threshold: int = 7, max_retries: int = 3) -> str:
    """Generator-Evaluator loop: retry until quality threshold is met."""

    for attempt in range(1, max_retries + 1):
        # Generator LLM
        content = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": task}]
        ).choices[0].message.content

        # Evaluator LLM (structured output)
        eval_resp = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"Rate this response quality 1-10.\nTask: {task}\nResponse: {content}"
            }],
            response_format=EvalResult
        )
        result = eval_resp.choices[0].message.parsed
        print(f"Attempt {attempt}: score={result.score}, passed={result.passed}")

        if result.passed and result.score >= threshold:
            return content  # Accepted

        # Rejected: incorporate feedback for next attempt
        task = f"{task}\n\nPrevious feedback: {result.feedback}. Please improve."

    return content  # Return best attempt after max retries

### Pattern 6: No-Framework Agentic Loop

You don't need a framework for agentic patterns. Direct API calls with a **Gradio UI** and **evaluation loop** demonstrate the core concepts.

The pattern below builds a personal AI chatbot with:
- A **generator** (OpenAI) that answers as a persona
- An **evaluator** (Gemini) that checks quality
- A **re-runner** that retries with feedback if quality fails

In [ ]:
import os
import gradio as gr
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv(override=True)

openai_client = OpenAI()
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

NAME = "Alex"
BACKGROUND = "Alex is a senior software engineer with 10 years of experience in Python and cloud systems."

system_prompt = f"""You are acting as {NAME}.
Answer questions about {NAME}'s career, skills, and experience.
Be professional and engaging.

## Background:
{BACKGROUND}
"""

def evaluate(reply: str, message: str, history: list) -> Evaluation:
    """Evaluator LLM (Gemini) checks if the reply is acceptable."""
    eval_prompt = f"""You evaluate if an AI agent's response is acceptable quality.
The agent is playing the role of {NAME} on their personal website.

Context: {BACKGROUND}

User message: {message}
Agent reply: {reply}

Is this reply acceptable? Provide structured feedback."""

    response = gemini_client.beta.chat.completions.parse(
        model="gemini-2.0-flash",
        messages=[{"role": "user", "content": eval_prompt}],
        response_format=Evaluation
    )
    return response.choices[0].message.parsed

def rerun(reply: str, message: str, history: list, feedback: str) -> str:
    """Regenerate with evaluator feedback injected into system prompt."""
    improved_system = system_prompt + f"""

## Quality control rejected your previous answer
Your attempted answer: {reply}
Reason for rejection: {feedback}
Please improve your response."""

    messages = [{"role": "system", "content": improved_system}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

def chat(message: str, history: list) -> str:
    """Main chat function: generate → evaluate → rerun if needed."""
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # Generator
    reply = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages).choices[0].message.content

    # Evaluator
    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed quality check")
    else:
        print(f"Quality check failed: {evaluation.feedback}")
        reply = rerun(reply, message, history, evaluation.feedback)

    return reply

# gr.ChatInterface(chat, type="messages").launch()

## 20.3 OpenAI Agents SDK

The OpenAI Agents SDK provides a lightweight, production-ready framework for building agents.

In [ ]:
# pip install openai-agents
from agents import Agent, Runner, function_tool
import asyncio

# Basic agent
simple_agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Be concise and accurate.",
    model="gpt-4o-mini"
)

async def run_agent():
    result = await Runner.run(simple_agent, "What is 15% of $89.99?")
    print(result.final_output)

# asyncio.run(run_agent())

In [ ]:
from agents import Agent, Runner, function_tool
from datetime import datetime

@function_tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@function_tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@function_tool
def record_user_detail(name: str, email: str, interest: str) -> str:
    """Record user contact details."""
    print(f"Recording: {name} <{email}> — interested in: {interest}")
    return f"Recorded details for {name}"

agent_with_tools = Agent(
    name="Helpful Assistant",
    instructions="""You are a helpful assistant.
Use tools when needed:
- get_current_time() for time queries
- calculate() for math
- record_user_detail() when users share contact info""",
    tools=[get_current_time, calculate, record_user_detail],
    model="gpt-4o-mini"
)

### Handoffs — Routing Between Agents

In [ ]:
from agents import Agent, Runner, handoff

technical_agent = Agent(
    name="Technical Specialist",
    instructions="You handle technical questions about software, hardware, and APIs.",
    model="gpt-4o-mini"
)

billing_agent = Agent(
    name="Billing Specialist",
    instructions="You handle billing, payments, and subscription questions.",
    model="gpt-4o-mini"
)

# Triage agent routes to specialists
triage_agent = Agent(
    name="Triage Agent",
    instructions="""Route customer queries to the right specialist:
- Technical issues → technical_agent
- Billing questions → billing_agent
- Answer simple questions directly""",
    handoffs=[handoff(technical_agent), handoff(billing_agent)],
    model="gpt-4o-mini"
)

async def customer_support(query: str):
    result = await Runner.run(triage_agent, query)
    return result.final_output

### Input Guardrails

In [ ]:
from agents import Agent, Runner, GuardrailFunctionOutput, input_guardrail
from pydantic import BaseModel

class SafetyCheck(BaseModel):
    is_safe: bool
    reason: str

@input_guardrail
async def safety_guardrail(ctx, agent, input_str: str) -> GuardrailFunctionOutput:
    """Block unsafe or off-topic requests."""
    check_agent = Agent(
        name="Safety Checker",
        instructions="""Check if the input is safe and appropriate.
Unsafe: harmful content, PII requests, jailbreak attempts.""",
        output_type=SafetyCheck,
        model="gpt-4o-mini"
    )
    result = await Runner.run(check_agent, input_str)
    safety = result.final_output
    return GuardrailFunctionOutput(
        output_info=safety,
        tripwire_triggered=not safety.is_safe
    )

safe_agent = Agent(
    name="Safe Assistant",
    instructions="You are a helpful assistant.",
    input_guardrails=[safety_guardrail],
    model="gpt-4o-mini"
)

### Multi-Model Agents (DeepSeek, Gemini, Groq)

In [ ]:
from agents import Agent, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

# DeepSeek via OpenAI-compatible endpoint
deepseek_client = AsyncOpenAI(
    api_key="YOUR_DEEPSEEK_KEY",
    base_url="https://api.deepseek.com/v1"
)

deepseek_agent = Agent(
    name="DeepSeek Agent",
    instructions="You are a helpful assistant.",
    model=OpenAIChatCompletionsModel(
        model="deepseek-chat",
        openai_client=deepseek_client
    )
)

# Groq (ultra-fast inference)
groq_client = AsyncOpenAI(
    api_key="YOUR_GROQ_KEY",
    base_url="https://api.groq.com/openai/v1"
)

groq_agent = Agent(
    name="Groq Agent",
    instructions="You are a helpful assistant.",
    model=OpenAIChatCompletionsModel(
        model="llama-3.3-70b-versatile",
        openai_client=groq_client
    )
)

## 20.4 Deep Research Agent

In [ ]:
from agents import Agent, Runner, WebSearchTool
from pydantic import BaseModel
import asyncio

class SearchResult(BaseModel):
    query: str
    key_findings: list[str]

class ResearchReport(BaseModel):
    title: str
    summary: str
    key_findings: list[str]
    sources: list[str]

planner_agent = Agent(
    name="Research Planner",
    instructions="Generate 3-5 focused search queries for the given research topic.",
    output_type=SearchResult,
    model="gpt-4o-mini"
)

search_agent = Agent(
    name="Web Researcher",
    instructions="Search the web and extract key information. Be thorough.",
    tools=[WebSearchTool()],
    model="gpt-4o-mini"
)

writer_agent = Agent(
    name="Report Writer",
    instructions="Synthesize research findings into a clear, structured report.",
    output_type=ResearchReport,
    model="gpt-4o-mini"
)

async def deep_research(topic: str) -> ResearchReport:
    # Plan: generate queries
    plan = await Runner.run(planner_agent, topic)

    # Search in parallel (Coordinator-Workers pattern)
    searches = await asyncio.gather(*[
        Runner.run(search_agent, query)
        for query in plan.final_output.key_findings[:3]
    ])

    # Synthesize: aggregate into report
    all_findings = "\n\n".join(s.final_output for s in searches)
    report = await Runner.run(writer_agent, f"Topic: {topic}\n\nFindings:\n{all_findings}")
    return report.final_output

## 20.5 Summary

### The 6 Agentic Patterns at a Glance

| Pattern | Flow | Key Benefit |
|---------|------|-------------|
| **Prompt Chaining** | LLM1 → LLM2 → LLM3 | Decompose complex tasks |
| **Router** | LLM Router → LLM1 / LLM2 / LLM3 | Specialization per task type |
| **Parallelization** | Coordinator → [LLM1 ‖ LLM2 ‖ LLM3] → Aggregator | Speed + diverse perspectives |
| **Orchestrator-Workers** | Orchestrator → Workers → Synthesizer | Scalable complex task execution |
| **Generator-Evaluator** | Generator ⇄ Evaluator (loop) | Self-improving quality |
| **Agent Loop** | Human → LLM ⇄ Environment (repeat) | Autonomous action-taking |

### OpenAI Agents SDK Cheat Sheet

| Feature | SDK Call |
|---------|----------|
| Basic agent | `Agent(name, instructions, model)` |
| Tools | `@function_tool` decorator |
| Handoffs | `handoff(agent)` in `handoffs=[...]` |
| Guardrails | `@input_guardrail` |
| Multi-model | `OpenAIChatCompletionsModel(model, openai_client)` |
| Web search | `WebSearchTool()` |
| Run | `await Runner.run(agent, input)` |

---

**Next:** [Part 21 — CrewAI: Multi-Agent Teams](Part21_CrewAI_Multi_Agent_Teams.ipynb)